In [57]:
import cv2
import mediapipe as mp
import numpy as np
import matplotlib.pyplot as plt
import os
import urllib
from mediapipe.tasks import python
from mediapipe.tasks.python import vision
from mediapipe import Image, ImageFormat

# Accessing Webcam

In [ ]:
# video_capture=cv2.VideoCapture(0)
# while True:
#     ret,video_data=video_capture.read()
#     cv2.imshow("video_live",video_data)
#     if (cv2.waitKey(10)==ord('a')):
#         break
# video_capture.release()    
# cv2.destroyAllWindows()

In [ ]:
video_cap=cv2.VideoCapture(0)
# 0 -> opens the default cam
while True:
    ret, video_data=video_cap.read()
    cv2.imshow("Umyma's Video",video_data)
    # waitkey pauses the program briefly anc captures input
    if(cv2.waitKey(20)==ord("c")):
        break
video_cap.release() 
# kernel crashes if the window is not destroyed
cv2.destroyAllWindows()

# Face Detection

In [ ]:
face_cap=cv2.CascadeClassifier(cv2.data.haarcascades + "haarcascade_frontalface_default.xml")
video_capture=cv2.VideoCapture(0)
while True:
    ret, vid_data=video_capture.read()
    col=cv2.cvtColor(vid_data,cv2.COLOR_BGR2GRAY)
    face=face_cap.detectMultiScale(
        col,
        scaleFactor=1.3,
        minNeighbors=5,
        minSize=(30,30),
        flags=cv2.CASCADE_SCALE_IMAGE)
    # print("Faces detected:", len(face))
    for (x,y,w,h) in face:
        cv2.rectangle(vid_data,(x,y),(x+w,y+h),(0,255,0),2)
    cv2.imshow("Video",vid_data)
    if (cv2.waitKey(10)==ord("a")):
        # break if "a" is pressed for 10 mil secs
        break
video_capture.release()
cv2.destroyAllWindows()   

Landmark Extraction

In [91]:
model_path = "face_landmarker.task"
if not os.path.exists(model_path):
    urllib.request.urlretrieve(
        "https://storage.googleapis.com/mediapipe-models/face_landmarker/face_landmarker/float16/1/face_landmarker.task",
        model_path
    )

In [40]:
#  creating a facelandmarker object

base_options=python.BaseOptions(model_asset_path=model_path)
options=vision.FaceLandmarkerOptions(base_options=base_options,
                                    output_face_blendshapes=True, #detects facial expressions
                                    output_facial_transformation_matrixes=True, #calculates 3d position of the face in space
                                    num_faces=1) #looks for only one face at a time
detector=vision.FaceLandmarker.create_from_options(options=options)
# detector object refers to the pretrained model

In [56]:
# getting frames from webcam
cap=cv2.VideoCapture(0) #captures 30 frames/sec
while True:
    ret,frame=cap.read() #frame contains the caotured img as an np array
    frame=cv2.flip(frame,1)
    # flipping the frame horizontally (mirrored view)
    rgb_frame=cv2.cvtColor(frame,cv2.COLOR_BGR2RGB) #swapping the color channel order
    # cv2 reads images in gbr order whereas mp expects rgb
    mp_image=Image(image_format=ImageFormat.SRGB,data=rgb_frame) 
    # np img array -> mp img obj
    results=detector.detect(mp_image)
    if results.face_landmarks:
        h,w=frame.shape[:2]
        # gets the h and w of the frame
        for face_landmarks in results.face_landmarks:
            # loops over the detected face (1)
            for landmark in face_landmarks:
                # loops over 478 landmark points
                x = int(landmark.x * w)
                y = int(landmark.y * h)
                cv2.circle(frame, (x, y), 1, (0, 255, 0), -1)
    cv2.imshow("Face Mesh",frame)
    if cv2.waitKey(10)==ord('a'):
        break            
cap.release()
detector.close()
cv2.destroyAllWindows()


In [92]:
# Haar cascade for face detetction
face_cascade=cv2.CascadeClassifier(cv2.data.haarcascades + "haarcascade_frontalface_default.xml")
base_options = python.BaseOptions(model_asset_path=model_path)
options = vision.FaceLandmarkerOptions(
    base_options=base_options,
    num_faces=1,
    min_face_detection_confidence=0.2,
    min_tracking_confidence=0.2
)
detector = vision.FaceLandmarker.create_from_options(options)

In [93]:
def align_faces(image,left_eye,right_eye):
    dx=right_eye[0]-left_eye[0]
    dy=right_eye[1]-left_eye[1]
    angle=np.degrees(np.arctan2(dy,dx))
    center=((left_eye[0]+right_eye[0])//2, (left_eye[1]+right_eye[1])//2)
    m=cv2.getRotationMatrix2D(center,angle,1)
    rotated=cv2.warpAffine(image,m,(image.shape[1],image.shape[0]))
    return rotated

In [ ]:
images = os.listdir("images")[:4]

for i, file in enumerate(images):
    path = os.path.join("images", file)
    # using imdecode instead of imread since our folder has images with webp extension aswell
    img = cv2.imdecode(np.fromfile(path, dtype=np.uint8), cv2.IMREAD_COLOR)
   
    rgb  = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    # running mediapipe on full image
    mp_image = Image(image_format=ImageFormat.SRGB, data=rgb)
    results  = detector.detect(mp_image)
    
    h_img, w_img = rgb.shape[:2]
    points = []
    for lm in results.face_landmarks[0]:
        px = int(lm.x * w_img)
        py = int(lm.y * h_img)
        points.append((px, py))

    left_eye  = points[33]
    right_eye = points[263]

    # aligning full img
    aligned_full = align_faces(rgb, left_eye=left_eye, right_eye=right_eye)

    # drawing landmarks on original for visualization
    vis = rgb.copy()
    for (px, py) in points:
        cv2.circle(vis, (px, py), 1, (0, 255, 0), -1)

    # try haar crop on aligned image
    gray_aligned = cv2.cvtColor(aligned_full, cv2.COLOR_RGB2GRAY)
    haar_faces   = face_cascade.detectMultiScale(gray_aligned, 1.3, 5)

    if len(haar_faces) == 0:
        cx = (left_eye[0] + right_eye[0]) // 2
        cy = (left_eye[1] + right_eye[1]) // 2
        eye_dist  = int(np.linalg.norm(np.array(right_eye) - np.array(left_eye)))
        crop_size = int(eye_dist * 3.5)
        x1 = max(cx - crop_size // 2, 0)
        y1 = max(cy - crop_size // 2, 0)
        x2 = min(cx + crop_size // 2, w_img)
        y2 = min(cy + crop_size // 2, h_img)
        face_crop = aligned_full[y1:y2, x1:x2]
    else:
        x, y, w, h = haar_faces[0]
        face_crop  = aligned_full[y:y+h, x:x+w]

    if face_crop.size == 0:
        print(f"Empty crop: {file}")
        continue

    face_crop = cv2.resize(face_crop, (256, 256))
    save_path = os.path.join("aligned_faces", f"aligned_{i:02d}_{file.split('.')[0]}.jpg")
    cv2.imwrite(save_path, cv2.cvtColor(face_crop, cv2.COLOR_RGB2BGR))

    plt.figure(figsize=(8, 4))
    plt.subplot(1, 2, 1)
    plt.title(f"Landmarks — {file}")
    plt.imshow(vis)
    plt.axis("off")
    plt.subplot(1, 2, 2)
    plt.title("Aligned Face (256x256)")
    plt.imshow(face_crop)
    plt.axis("off")
    plt.tight_layout()
    plt.show()

detector.close()
print("Task Completed")